---
title: "Лабораторна робота №3. Обчислення ентропійних характеристик інформаційної системи"
description: "Моніторинг та керування в слабкоструктурованих процесах і системах | КрНУ ім. М. Остроградського"
author: "Ярослав Фігас, 2026"
date: today
lang: uk
jupyter: python3

format:
  html:
    toc: true
    toc-location: right
    number-sections: false
    code-fold: true
    embed-resources: true
    self-contained-math: true
---

# Мета роботи
Опанувати підходи до *кількісного оцінювання невизначеності* в інформаційних системах та навчитися:
- обчислювати *максимальну* і *реальну* ентропію дискретного джерела;
- порівнювати ентропії різних джерел інформації;
- знаходити *часткову* та *повну умовну ентропію*;
- обчислювати *взаємну інформацію* між випадковими величинами;
- інтерпретувати одержані результати у задачах *моніторингу станів системи*.

# Індивідуальне завдання

## Умови роботи

Потрібно виконати наступні дії:

1. Взяти один із наведених у методичних вказівках варіантів чи узгодити свій приклад із викладачем.
2. Обчислити:
- максимальну ентропію;
- реальну ентропію;
- умовну ентропію;
- взаємну інформацію.
3. Пояснити, що означають ці значення в контексті конкретної інформаційної системи:
- виробничої;
- організаційної;
- інформаційно-аналітичної;
- системи моніторингу повідомлень, подій або станів.


## Виконання роботи

Для початку роботи потрібно імпортувати базові імпорти, наступним кроком задати власний розподіл ймовірностей, а також умовний розподіл.

In [52]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

names = ["Норма", "Попередження", "Критична ситуація"]
my_probabilities = np.array([0.75, 0.20, 0.05])
my_p_y_given_x = np.array([
    [0.9, 0.05, 0.05],
    [0.15, 0.75, 0.1],
    [0.05, 0.15, 0.8]
])

Наступним кроком буде розрахунок максимальної та реальної ентропії. З результатів бачимо що кількість реальної ентропії складає 62% від максимально можливої.

In [53]:
def validate_probabilities(probabilities, tol=1e-9):
    probs = np.array(probabilities, dtype=float)
    if np.any(probs < 0):
        raise ValueError("Ймовірності не можуть бути від'ємними.")
    total = probs.sum()
    if not np.isclose(total, 1.0, atol=tol):
        raise ValueError(f"Сума ймовірностей має дорівнювати 1. Зараз: {total}")
    return probs

def hartley_entropy(m):
    return math.log2(m)

def shannon_entropy(probabilities):
    probs = validate_probabilities(probabilities)
    nonzero = probs[probs > 0]
    return -np.sum(nonzero * np.log2(nonzero))

h_max = hartley_entropy(len(my_probabilities))
h_real = shannon_entropy(my_probabilities)

print("Максимальна ентропія дорівнює:", h_max)
print("Реальна ентропія дорівнює:", h_real)
print("Відсоток реальної ентропії від максимальної:", h_real/h_max)

Максимальна ентропія дорівнює: 1.584962500721156
Реальна ентропія дорівнює: 0.9917601481809735
Відсоток реальної ентропії від максимальної: 0.6257309858938137


Далі буде проводитись розрахунок часткова та повна умовна ентропія. Для цього знадобиться умовний розподіл, який описали на початку. Як можна помітити найбільша часткова ентропія відбувається на попередженні, але беручи уваги коефіцієнти, то найбільша ентропія припадає саме на норму. Повна умовна ентропія дорівнює 0.681759 біт, а це майже 69% від реальної ентропії.

In [54]:
def conditional_entropy_from_conditional(px, p_y_given_x):
    px = validate_probabilities(px)
    cond = np.array(p_y_given_x, dtype=float)
    if cond.shape[0] != len(px):
        raise ValueError("Кількість рядків матриці умовних ймовірностей має збігатися з розміром p(x).")
    for row in cond:
        validate_probabilities(row)
    value = 0.0
    partial = []
    for i, row in enumerate(cond):
        h_row = shannon_entropy(row)
        partial.append(h_row)
        value += px[i] * h_row
    return value, np.array(partial)
    
h_y_given_x, partial_entropies = conditional_entropy_from_conditional(my_probabilities, my_p_y_given_x)

partial_df = pd.DataFrame({
    "Стан X": names,
    "Часткова умовна ентропія H(Y|x_i)": partial_entropies,
    "Вага p(x_i)": my_probabilities,
    "Внесок у повну H(Y|X)": partial_entropies * my_probabilities
})
partial_df

,Стан X,Часткова умовна ентропія H(Y|x_i),Вага p(x_i),Внесок у повну H(Y|X)
0,Норма,0.568996,0.75,0.426747
1,Попередження,1.054016,0.20,0.210803
2,Критична ситуація,0.884184,0.05,0.044209


In [55]:

print(f"Повна умовна ентропія H(Y|X) = {h_y_given_x:.4f} біт")

print(f"Відсоток повної умовної ентропії від реальної: {(h_y_given_x/h_real*100):.4f}%")

Повна умовна ентропія H(Y|X) = 0.6818 біт
Відсоток повної умовної ентропії від реальної: 68.7423%


На останок знайдемо взаємну інформацію яка скільки інформації про поточний стан ми отримуємо завдяки знанню попереднього стану. Для початку потрібно вирахувати безумовний розподіл, вже після цього знайдемо потрібну величину.

In [56]:
def joint_distribution(px, p_y_given_x):
    px = validate_probabilities(px)
    cond = np.array(p_y_given_x, dtype=float)
    joint = px[:, None] * cond
    return joint

joint = joint_distribution(my_probabilities ,my_p_y_given_x)
p_y = joint.sum(axis=0)

joint_df = pd.DataFrame(
    joint,
    index=names,
    columns=names
)

joint_df

,Норма,Попередження,Критична ситуація
Норма,0.6750,0.0375,0.0375
Попередження,0.0300,0.1500,0.0200
Критична ситуація,0.0025,0.0075,0.0400


In [57]:
py_df = pd.DataFrame({
    "Стан Y": names,
    "Ймовірність": p_y
})

py_df

,Стан Y,Ймовірність
0,Норма,0.7075
1,Попередження,0.1950
2,Критична ситуація,0.0975


In [58]:
def entropy_from_joint(joint):
    probs = np.array(joint, dtype=float).ravel()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def mutual_information(px, py, joint=None, p_y_given_x=None):
    px = validate_probabilities(px)
    py = validate_probabilities(py)
    hx = shannon_entropy(px)
    hy = shannon_entropy(py)
    if joint is None:
        if p_y_given_x is None:
            raise ValueError("Потрібно задати або joint, або p_y_given_x.")
        joint = joint_distribution(px, p_y_given_x)
    hxy = entropy_from_joint(joint)
    ixy = hx + hy - hxy
    return hx, hy, hxy, ixy

h_x, h_y, h_xy, i_xy = mutual_information(my_probabilities, p_y, joint=joint)
print(f"Кількість взаємної інформації дорівнює: {i_xy}")

Кількість взаємної інформації дорівнює: 0.45877129815848017


# Висновки

У цій лабораторній роботі працювали з підходами кількісного оцінювання невизначеності, а саме:  ентропії Хартлі та Шеннона, умовної ентропії та взаємної інформації.
Під час виконання роботи було виявлено, що реальна ентропія доволі відчутно менша за максимальну через нерівномірний розподіл станів, умовна ентропія показала значну невизначеність, а кількість взаємної інформації показує дуже гарний результат порівнюючи з кількістю реальної ентропії.
За допомогою ентропійних характеристиких отримуємо можливість аналізувати невизначеності та оцінювати якість інформації.

# Контрольні питання

1. Що таке *кількість інформації* та *ентропія*?

Ентропія відповідає на запитання: скільки “середньої інформації” міститься в одному повідомленні джерела або еквівалентно: яка середня невизначеність щодо того, що видасть джерело на наступному кроці. Кількість інформації — це міра зменшення невизначеності (або міра здивування) від отримання нового повідомлення.

2. У чому полягає відмінність між *формулою Хартлі* та *формулою Шеннона*?

Формула Хартлі рахує інформацію для ідеальних умов, а формула Шеннона також враховує ймовірності настання певних подій

3. Чому за нерівномірного розподілу ентропія зменшується?

Ентропія зменшується у тому випадку, коли невизначеності стає менше, тобто ймовірності певних подій більше за інші

4. Що показує *умовна ентропія*?

Ця величина показує, скільки невизначеності залишається щодо певної величини, якщо інформація про іншу величину уже доступна.

5. Що таке *часткова умовна ентропія* і чим вона відрізняється від повної?

Часткова умовна ентропія показує невизначеність поточного стану, якщо конкретний попередній стан уже відомий, в той час як повна це середнє значення всіх часткових умовних ентропій.

6. Що характеризує *взаємна інформація*?

Взаємна інформація показує, скільки інформації в поточному стані ми отримуємо завдяки знанню попереднього стану.

7. Коли взаємна інформація дорівнює нулю?

Взаємна інформація дорівнює нулю тоді, коли попередній стан не надає нових знань поточному стану.

8. Як ентропійні характеристики можна використовувати у задачах *моніторингу та керування*?

Ентропійні характеристики можна використовувати для оцінки невизначеності у системах.